In [ ]:
import numpy as np
import pandas as pd
import sys
import torch
sys.path.insert(0, '../al_framework/al_methods')
import mqr_ud
from evaluation import Evaluation
from Probability import Probability
sys.path.insert(0, '../al_framework/batch_split')
import BatchGenerator

In [ ]:
# Hyper-Params:
iteration = 20  # Repeated Experiments
n_queries = 10
batch_size = 10  # AL Batch Size
B = 5  # How many batches need to be queried in each AL iteration
initial_size = 10  # Samples for initalization
seed_rf = np.load(file="../../seed_for_model.npy")  # Seeds for model initialization
seed_initial = np.load(file="../../seed_for_initialSamples.npy")     # Seed for AL initial samples
device = torch.device("cpu")
metric = "EP"
Q = 100
K = 5
model_size = 1000
model_type = "Tree"
quantiles = [q/Q for q in range(1, Q)]
path_data = "../../datasets/Graphene_Periodic/split_data/"
path_res = "Graphene_Periodic/5_Batch_Res_F64"
path_method = "/mqr_ud"

In [ ]:
data_loader = BatchGenerator.PeriodicGraphene()
evaluator = Evaluation(quantiles, K, Q)
al_learner = mqr_ud.ActiveLearner(Q, K, n_queries, metric, initial_size, quantiles, batch_size, B, device)
calculator = Probability(Q, model_size, quantiles, model_type, device)

# For each AL experiment:
for i in range(iteration):
    print("Iteration:", i)
    X_train_ = pd.read_csv(path_data + "X_train" + str(i) + ".csv")
    X_test_ = pd.read_csv(path_data + "X_test" + str(i) + ".csv")
    y_train_ = pd.read_csv(path_data + "y_train" + str(i) + ".csv")
    y_test_ = pd.read_csv(path_data + "y_test" + str(i) + ".csv")

    y_train_ = y_train_['Fermi_energy'].to_numpy().reshape(-1,1)
    y_test_ = y_test_['Fermi_energy'].to_numpy().reshape(-1,1)
    
    testing_r2, testing_mse, used_data, used_label, acquisition_values, acquisition_selected = al_learner.al_iterations(data_loader,
    X_train_, X_test_, y_train_, y_test_, seed_initial[i], seed_rf[i], evaluator, calculator)

    file_name_1_ = "../../results/"+ path_res + path_method +"/Summary/averaged_test_r2_"
    file_name_2_ = "../../results/"+ path_res + path_method +"/Summary/averaged_test_mse_"

    np.save(file=file_name_1_+ str(i) + ".npy", arr=testing_r2)
    np.save(file=file_name_2_+ str(i) + ".npy", arr=testing_mse)

    file_name_1_1 = "../../results/"+ path_res + path_method +"/Summary/used_data_"
    file_name_2_1 = "../../results/"+ path_res + path_method +"/Summary/used_label_"
    np.save(file=file_name_1_1+ str(i) + ".npy", arr=used_data)
    np.save(file=file_name_2_1+ str(i) + ".npy", arr=used_label)
    
    file_name_3_1 = "../../results/"+ path_res + path_method +"/Summary/acquisition_values_"
    np.save(file=file_name_3_1+ str(i) + ".npy", arr=np.array(acquisition_values, dtype=object), allow_pickle=True)
    file_name_3_2 = "../../results/"+ path_res + path_method +"/Summary/acquisition_selected_"
    np.save(file=file_name_3_2+ str(i) + ".npy", arr=np.array(acquisition_selected, dtype=object), allow_pickle=True)